# 02 — Ingestion & Extraction


> **Note.** This notebook shows the extraction pipeline internals via the legacy `InfonEngine` class. The cassette-native path (`InfonStore.ingest` + `extraction_report`) ships the same extractor but adds automatic coverage diagnostics and delta-append semantics. See **[00 — Quick Start](00_quick_start.ipynb)** for the current user-facing API.

What happens inside `cog.ingest()`:

```
documents → split sentences → SPLADE encode → project onto anchors → form triples → score → store
```

This notebook unpacks each stage so you can see the internal representations.

In [ ]:
import json
import numpy as np
from pathlib import Path
from infon import InfonEngine, InfonConfig
from infon.schema import AnchorSchema
from infon.encoder import Encoder, SpladeEncoder, AnchorProjector
from infon.extract import split_sentences, extract_infons

In [ ]:
SCHEMA = {
    "toyota":      {"type": "actor", "tokens": ["toyota"]},
    "tesla":       {"type": "actor", "tokens": ["tesla"]},
    "ford":        {"type": "actor", "tokens": ["ford"]},
    "panasonic":   {"type": "actor", "tokens": ["panasonic"]},
    "invest":      {"type": "relation", "tokens": ["invest", "investment", "investing"]},
    "launch":      {"type": "relation", "tokens": ["launch", "unveil", "introduce"]},
    "partner":     {"type": "relation", "tokens": ["partner", "partnership", "collaborate"]},
    "expand":      {"type": "relation", "tokens": ["expand", "expansion", "grow"]},
    "battery":     {"type": "feature", "tokens": ["battery", "batteries"]},
    "ev":          {"type": "feature", "tokens": ["ev", "electric vehicle", "electric"]},
    "solid_state": {"type": "feature", "tokens": ["solid-state", "solid state"]},
    "autonomous":  {"type": "feature", "tokens": ["autonomous", "self-driving"]},
    "us":          {"type": "market", "tokens": ["us", "united states", "america"]},
    "europe":      {"type": "market", "tokens": ["europe", "european"]},
    "china":       {"type": "market", "tokens": ["china", "chinese"]},
}

schema = AnchorSchema(SCHEMA)
encoder = Encoder(schema=schema)
print(f"Schema: {len(schema.names)} anchors, Encoder device: {encoder.device}")

## Stage 1: Sentence splitting

Documents are split into sentences. Each sentence becomes one encoding unit.

In [ ]:
doc_text = """Toyota announced a $13.5 billion investment in solid-state battery technology. The Japanese automaker plans to begin mass production by 2027, targeting a 50% increase in EV range. Toyota partnered with Panasonic to scale manufacturing in Europe."""

sentences = split_sentences(doc_text)
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

## Stage 2: SPLADE sparse encoding

Each sentence goes through the SPLADE model (BertForMaskedLM):

```
tokens → BERT → MLM logits → log(1 + ReLU(logits)) → max-pool → 30,522-dim sparse vector
```

The result is a sparse vector over BERT's entire vocabulary. High values mean
the model strongly associates that vocabulary word with the input.

In [ ]:
sparse_vecs = encoder.encode_sparse(sentences)
print(f"Shape: {sparse_vecs.shape}  (sentences × vocab_size)")

for i, sent in enumerate(sentences):
    vec = sparse_vecs[i]
    nnz = np.count_nonzero(vec)
    print(f"\n  [{i}] \"{sent[:60]}...\"")
    print(f"      nonzero: {nnz}/{vec.shape[0]}  max: {vec.max():.3f}  mean(>0): {vec[vec>0].mean():.3f}")
    
    # Top vocabulary activations
    top_ids = np.argsort(vec)[-10:][::-1]
    tokens = [encoder.tokenizer.convert_ids_to_tokens(int(tid)) for tid in top_ids]
    scores = [vec[tid] for tid in top_ids]
    for tok, sc in zip(tokens, scores):
        print(f"      {tok:20s} {sc:.3f}")

## Stage 3: Anchor projection

The AnchorProjector maps the 30,522-dim SPLADE vector to your schema's anchors.
For each anchor, it takes the **max** of the SPLADE scores at that anchor's token IDs.

```
anchor_score = max(sparse_vec[token_id] for token_id in anchor.token_ids)
```

This is the key insight: no training needed. SPLADE learned broad vocabulary coverage
from 8.8M passages. We just look up the tokens we care about.

In [ ]:
anchor_matrix = encoder.encode(sentences)
print(f"Shape: {anchor_matrix.shape}  (sentences × anchors)")

for i, sent in enumerate(sentences):
    print(f"\n  [{i}] \"{sent[:60]}...\"")
    scores = anchor_matrix[i]
    for j, name in enumerate(encoder.anchor_names):
        if scores[j] > 0.1:  # only show significant activations
            atype = SCHEMA[name]["type"]
            bar = '█' * int(scores[j] * 15)
            print(f"      {atype:10s} {name:15s} {scores[j]:.3f} {bar}")

## Stage 4: Triple formation

Activated anchors are partitioned by role:
- `actor` anchors → subjects
- `relation` anchors → predicates
- everything else → objects

The top-k per role form a cartesian product. Each combination becomes a candidate infon.
The confidence score is the geometric mean of the three role probabilities (after normalization).

In [ ]:
config = InfonConfig(schema_path="unused", activation_threshold=0.3, top_k_per_role=3)

documents = [
    {"id": "test-001", "timestamp": "2024-03-15",
     "text": doc_text},
]

infons, edges = extract_infons(documents, encoder, schema, config)

print(f"Extracted {len(infons)} infons, {len(edges)} edges\n")

for inf in infons[:10]:
    pol = "+" if inf.polarity else "-"
    grounding = []
    for role in ["subject", "predicate", "object"]:
        g = inf.support.get(role, "?")[0].upper()
        grounding.append(g)
    g_str = "|".join(grounding)
    print(f"  {pol}<<{inf.predicate}, {inf.subject}, {inf.object}>>  [{g_str}]  conf={inf.confidence:.3f}")
    print(f"    \"{inf.sentence[:70]}...\"")

## Stage 5: Grounding and metadata

Each infon carries rich metadata beyond the triple itself:

In [ ]:
inf = infons[0]
print(f"Triple: <<{inf.predicate}, {inf.subject}, {inf.object}>>")
print(f"  polarity:     {inf.polarity} ({'affirmed' if inf.polarity else 'negated'})")
print(f"  confidence:   {inf.confidence:.3f}")
print(f"  importance:   {inf.importance:.3f}")
print(f"  tense:        {inf.tense}")
print(f"  timestamp:    {inf.timestamp}")
print(f"  doc_id:       {inf.doc_id}")
print(f"  sent_id:      {inf.sent_id}")
print()
print(f"  Support (grounding type per role):")
for role, stype in inf.support.items():
    print(f"    {role:12s} → {stype}")
print()
print(f"  Spans:")
for role, span in inf.spans.items():
    print(f"    {role:12s} → \"{span['text']}\" [{span['start']}:{span['end']}]")
print()
print(f"  Subject meta:   {inf.subject_meta}")
print(f"  Predicate meta: {inf.predicate_meta}")
print(f"  Object meta:    {inf.object_meta}")
print()
print(f"  Temporal refs:  {inf.temporal_refs}")
print(f"  Locations:      {inf.locations}")

## Stage 6: Spoke edges

Each infon generates spoke edges connecting it to its anchor roles:

```
subject ──INITIATES──→ infon ──ASSERTS──→ predicate
                         │
                    TARGETS
                         │
                         ▼
                       object
```

In [ ]:
# Show edges for the first infon
target_id = infons[0].infon_id
related_edges = [e for e in edges if e.source == target_id or e.target == target_id]

print(f"Edges for infon {target_id[:20]}...:")
for e in related_edges:
    src = e.source[:20] + "..." if len(e.source) > 20 else e.source
    tgt = e.target[:20] + "..." if len(e.target) > 20 else e.target
    print(f"  {src} ──{e.edge_type}──→ {tgt}  weight={e.weight:.3f}")

## Importance scoring

Each infon gets an importance score that determines its rank in query results:

```
importance = w_activation × confidence
           + w_coherence × coherence
           + w_specificity × idf_score
           + w_novelty × novelty
```

- **activation**: how confident the model is (geometric mean of role scores)
- **coherence**: sheaf consistency (computed during consolidation)
- **specificity**: inverse document frequency of the triple's anchors
- **novelty**: 1.0 for new triples, decreases with reinforcement

In [ ]:
print(f"{'Triple':50s} {'conf':>6s} {'spec':>6s} {'imp':>6s}")
print("-" * 72)
for inf in sorted(infons, key=lambda x: -x.importance)[:10]:
    triple = f"<<{inf.predicate}, {inf.subject}, {inf.object}>>"
    print(f"{triple:50s} {inf.confidence:6.3f} {inf.specificity:6.3f} {inf.importance:6.3f}")

## Quality filter — why it matters

Without any filter, a single messy sentence can explode into a Cartesian
product of dozens of candidate triples — most spurious. Infon ships
with three filters layered together:

1. **Joint-score floor** (`quality_threshold`) — minimum geometric mean
   of the per-role activations; triples that barely cross the activation
   threshold get cut.
2. **Role-type hard constraints** — the subject of an infon must be an
   `actor`, the predicate a `relation`, etc. No more "battery invests
   toyota."
3. **Per-sentence cap** (`max_triples_per_sentence`) — defence in depth
   against messy-sentence explosion. Candidates are scored, sorted, and
   the top-K kept.

These three filters together took the EV benchmark from **2,598 infons
extracted** (untracked) down to the tight 49 we see in the reasoner.


In [ ]:
# Demonstrate: same document, two ingest runs, only the threshold differs.
import tempfile, os, json
from infon import InfonEngine, InfonConfig

SCHEMA = {
    "toyota": {"type": "actor", "tokens": ["toyota"]},
    "tesla":  {"type": "actor", "tokens": ["tesla"]},
    "honda":  {"type": "actor", "tokens": ["honda"]},
    "invests":  {"type": "relation", "tokens": ["invest", "invests"]},
    "partners": {"type": "relation", "tokens": ["partner", "partners"]},
    "produces": {"type": "relation", "tokens": ["produce", "produces"]},
    "battery":  {"type": "feature", "tokens": ["battery", "batteries"]},
    "ev":       {"type": "feature", "tokens": ["ev", "electric vehicle"]},
    "japan":    {"type": "market",  "tokens": ["japan", "japanese"]},
}

with tempfile.TemporaryDirectory() as tmp:
    schema_path = os.path.join(tmp, "schema.json")
    with open(schema_path, "w") as f:
        json.dump(SCHEMA, f)

    doc = {"id": "d1",
           "text": "Toyota invests in battery technology in Japan. "
                   "Honda partners on EV battery. Tesla produces batteries."}

    for qt, cap in [(0.00, 10), (0.05, 2)]:
        cog = InfonEngine(InfonConfig(
            schema_path=schema_path,
            db_path=os.path.join(tmp, f"cog_{qt}.db"),
            quality_threshold=qt,
            max_triples_per_sentence=cap,
        ))
        n = cog.ingest([doc])
        print(f"quality_threshold={qt}  cap={cap}  →  {n} infons")
        cog.close()


## Incremental ingest + `cog.refresh()`

Ingest is append-only — you can keep adding documents. But the cached
reasoner was fitted on the *previous* graph, so its GNN weights are
stale. Call `cog.refresh()` to rebuild it against the current store
state.

Without a refresh, subsequent `reasoner()` calls still use the old
fitted weights — that's intentional so inference is fast by default,
but it can silently become stale.


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    schema_path = os.path.join(tmp, "schema.json")
    with open(schema_path, "w") as f:
        json.dump(SCHEMA, f)

    cog = InfonEngine(InfonConfig(
        schema_path=schema_path,
        db_path=os.path.join(tmp, "cog.db"),
        quality_threshold=0.05,
    ))
    cog.ingest([{"id": "d1", "text": "Toyota invests in battery."}])
    cog.consolidate()
    r1 = cog.reasoner()  # first fit
    print(f"after first ingest: {cog.stats()['infon_count']} infons")

    # Add a second document. The cached reasoner is now stale.
    cog.ingest([{"id": "d2", "text": "Tesla produces batteries."}])
    cog.consolidate()
    print(f"after second ingest: {cog.stats()['infon_count']} infons")

    summary = cog.refresh(verbose=True)
    print(f"\nrefresh summary: {summary}")
    cog.close()


---

**What you saw:**

1. Sentences split from document text
2. SPLADE produces sparse 30k-dim vocabulary vectors
3. `AnchorProjector` reduces to your schema's anchors via token-ID
   max-pooling
4. Activated anchors partitioned by type into (S, P, O) roles
5. Cartesian product + geometric-mean confidence → candidate infons
6. **Quality filter** (threshold + role-type + per-sentence cap) trims
   to the clean set that actually makes it into the store
7. Each infon carries spans, tense, polarity, evidentiality, modality,
   situation grounding
8. `cog.refresh()` rebuilds the cached reasoner after new ingests so
   queries see the new evidence
